# Compresión universal: LZ78

Los algoritmos de Lempel-Ziv (1977, 1978) son la base de la mayoría de los compresores modernos
(gzip, ZIP, PNG). Su propiedad fundamental es la **universalidad**: sin conocer la distribución
de la fuente, comprimen asintóticamente hasta la tasa de entropía.

Este cuaderno implementa LZ78 y estudia empíricamente su convergencia a la entropía.

**Artículos asociados:**
[Codificación de fuente](../../02-teoria-informacion/03-codificacion-de-fuente.md),
[Teorema de Shannon y capacidad](../../02-teoria-informacion/08-teorema-de-shannon-capacidad.md)

In [ ]:
import math
import random
from collections import Counter

def log2(x):
    return math.log2(x) if x > 0 else 0.0

print('Entorno listo.')

## El algoritmo LZ78

LZ78 construye un **diccionario incremental** de frases. El algoritmo recorre la entrada y:
1. Lee símbolos hasta encontrar una frase que no esté en el diccionario.
2. Emite un par `(índice del prefijo más largo conocido, nuevo símbolo)`.
3. Añade la frase al diccionario.

Al inicio el diccionario contiene solo la frase vacía (índice 0).

In [ ]:
def lz78_encode(sequence):
    """
    Codifica una secuencia con LZ78.
    Devuelve lista de pares (índice_prefijo, símbolo_nuevo).
    """
    dictionary = {'': 0}  # frase_vacía → índice 0
    current = ''
    output = []
    for symbol in sequence:
        extended = current + str(symbol)
        if extended in dictionary:
            current = extended
        else:
            # Emitir (índice del prefijo, símbolo nuevo)
            output.append((dictionary[current], symbol))
            dictionary[extended] = len(dictionary)
            current = ''
    # Residuo al final de la secuencia
    if current:
        output.append((dictionary[current], None))
    return output, dictionary

def lz78_bits(sequence):
    """
    Estima el número de bits necesarios para la salida LZ78.
    El par (k, s) necesita log2(k+1) bits para el índice + log2(|alfabeto|) para el símbolo.
    """
    encoded, dictionary = lz78_encode(sequence)
    alphabet = set(str(s) for s in sequence)
    sym_bits = log2(len(alphabet))
    total_bits = 0
    for k, (idx, sym) in enumerate(encoded):
        index_bits = log2(k + 1) if k > 0 else 1
        total_bits += index_bits + (sym_bits if sym is not None else 0)
    return total_bits

# Ejemplo simple
seq = list('aababcabcabc')
encoded, dic = lz78_encode(seq)
print('Ejemplo: codificación de "aababcabcabc"')
print('Pares emitidos:', encoded)
print('Diccionario final:')
for frase, idx in sorted(dic.items(), key=lambda x: x[1]):
    print(f'  [{idx}] "{frase}"')

## Convergencia a la tasa de entropía

El teorema de Ziv-Lempel establece que para cualquier fuente estacionaria ergódica,
la tasa de compresión de LZ78 converge casi seguramente a la tasa de entropía.

Verificamos esto empíricamente generando secuencias de longitud creciente
desde una fuente con distribución conocida.

In [ ]:
def generar_iid(n, probs, seed=42):
    """Genera una secuencia i.i.d. con distribución probs = {símbolo: probabilidad}."""
    rng = random.Random(seed)
    simbolos = list(probs.keys())
    pesos = [probs[s] for s in simbolos]
    return rng.choices(simbolos, weights=pesos, k=n)

def entropia(probs):
    """Entropía de Shannon de una distribución."""
    return -sum(p * log2(p) for p in probs.values() if p > 0)

def tasa_lz78(sequence):
    """Tasa de compresión LZ78 en bits por símbolo."""
    return lz78_bits(sequence) / len(sequence)

# Fuente 1: binaria sesgada p(0)=0.8
probs_sesgada = {'0': 0.8, '1': 0.2}
H_sesgada = entropia(probs_sesgada)

# Fuente 2: ternaria uniforme
probs_ternaria = {'a': 1/3, 'b': 1/3, 'c': 1/3}
H_ternaria = entropia(probs_ternaria)

print('Convergencia de LZ78 a la entropía (fuente binaria sesgada, p(0)=0.8):')
print(f'Entropía teórica: {H_sesgada:.4f} bits/símbolo')
print(f'{"Longitud":>10}  {"LZ78 (bits/símbolo)":>22}  {"Error":>10}')
for n in [100, 500, 2000, 10000, 50000]:
    seq = generar_iid(n, probs_sesgada)
    tasa = tasa_lz78(seq)
    print(f'{n:>10}  {tasa:>22.4f}  {abs(tasa - H_sesgada):>10.4f}')

print()
print('Convergencia de LZ78 (fuente ternaria uniforme):')
print(f'Entropía teórica: {H_ternaria:.4f} bits/símbolo')
print(f'{"Longitud":>10}  {"LZ78 (bits/símbolo)":>22}  {"Error":>10}')
for n in [100, 500, 2000, 10000, 50000]:
    seq = generar_iid(n, probs_ternaria)
    tasa = tasa_lz78(seq)
    print(f'{n:>10}  {tasa:>22.4f}  {abs(tasa - H_ternaria):>10.4f}')

## LZ78 con fuente con memoria (Markov)

Para una fuente con memoria, LZ78 converge a la **tasa de entropía** (entropía condicional),
que es menor que la entropía marginal. El diccionario captura implícitamente las correlaciones.

In [ ]:
def generar_markov(n, transicion, inicio=None, seed=42):
    """Genera una secuencia de una cadena de Markov."""
    rng = random.Random(seed)
    estados = list(transicion.keys())
    if inicio is None:
        estado = rng.choice(estados)
    else:
        estado = inicio
    seq = [estado]
    for _ in range(n - 1):
        nexts = list(transicion[estado].keys())
        pesos = list(transicion[estado].values())
        estado = rng.choices(nexts, weights=pesos)[0]
        seq.append(estado)
    return seq

def tasa_entropia_markov(transicion, iters=500):
    """Calcula la tasa de entropía de una cadena de Markov por iteración de potencias."""
    estados = list(transicion.keys())
    pi = {s: 1.0 / len(estados) for s in estados}
    for _ in range(iters):
        pi_new = {s: 0.0 for s in estados}
        for i in estados:
            for j, p in transicion[i].items():
                pi_new[j] += pi[i] * p
        pi = pi_new
    return sum(
        pi[i] * (-sum(p * log2(p) for p in transicion[i].values() if p > 0))
        for i in estados
    )

# Cadena de Markov binaria persistente (alta correlación)
p_cambio = 0.05
transicion = {
    '0': {'0': 1 - p_cambio, '1': p_cambio},
    '1': {'0': p_cambio,     '1': 1 - p_cambio},
}

H_rate = tasa_entropia_markov(transicion)
H_marginal = 1.0  # distribución estacionaria uniforme

print(f'Cadena Markov binaria persistente (p_cambio = {p_cambio}):')
print(f'  Entropía marginal H(X):          {H_marginal:.4f} bits/símbolo')
print(f'  Tasa de entropía H(X|X_prev):    {H_rate:.4f} bits/símbolo')
print()
print('Convergencia de LZ78 a la TASA de entropía:')
print(f'{"Longitud":>10}  {"LZ78 (bits/símbolo)":>22}  {"H_rate":>10}')
for n in [200, 1000, 5000, 20000]:
    seq = generar_markov(n, transicion)
    tasa = tasa_lz78(seq)
    print(f'{n:>10}  {tasa:>22.4f}  {H_rate:>10.4f}')

print()
print('LZ78 captura la correlación temporal y converge a H_rate < H_marginal.')

## Crecimiento del diccionario

El número de frases en el diccionario LZ78 crece como $n / \log n$ para una fuente i.i.d.,
donde $n$ es la longitud de la secuencia. Este ritmo sublineal garantiza que cada frase
necesita $\approx \log \log n$ bits de índice, lo que conduce a la compresión óptima.

In [ ]:
print('Crecimiento del diccionario LZ78 (fuente binaria uniforme):')
print(f'{"n":>8}  {"frases":>8}  {"n/log2(n)":>12}  {"frases/(n/log2n)":>18}')
for n in [100, 500, 2000, 10000, 50000]:
    seq = generar_iid(n, {'0': 0.5, '1': 0.5})
    _, dic = lz78_encode(seq)
    num_frases = len(dic) - 1  # excluir la frase vacía
    n_log_n = n / log2(n)
    ratio = num_frases / n_log_n
    print(f'{n:>8}  {num_frases:>8}  {n_log_n:>12.1f}  {ratio:>18.4f}')

print()
print('El ratio tiende a la entropía de la fuente (1.0 para la fuente uniforme).')

## Ideas clave

- LZ78 construye un diccionario sin conocer la distribución de la fuente: es **universal**.
- Para fuentes i.i.d., la tasa de compresión converge a $H(X)$ cuando $n \to \infty$.
- Para fuentes con memoria (Markov), converge a la tasa de entropía $H = H(X_{t+1}|X_t) \leq H(X)$.
- El diccionario crece como $n/\log n$; esto asegura que los índices de referencia sean cortos.
- La convergencia es lenta: se necesitan decenas de miles de símbolos para estar cerca de la entropía.
  Los compresores prácticos (gzip) usan variantes de LZ77 con ventanas deslizantes que aceleran la convergencia.